In [ ]:
"""
Cell 0 — Environment setup
Installs the LangChain ecosystem + Groq LLM provider (for all AI features),
FAISS (vector store, reserved for future RAG / skill-matching work),
and requests + BeautifulSoup (used by the portfolio scraper in Cell 6).
"""
!pip install -q langchain-core langchain-groq pypdf docx2txt scikit-learn requests beautifulsoup4

In [ ]:
"""
Cell 1 — API credentials

Loads the Groq API key from Colab's secret manager and exposes it as an
environment variable. ChatGroq (instantiated in Cell 3) reads GROQ_API_KEY
from the environment automatically.
"""
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
"""
Cell 2 — Shared imports

Standard-library and third-party imports used across the rest of the
notebook: enum/re/uuid/json/datetime/typing for the data models, pydantic
for schema validation, and the langchain_groq / langchain_core pieces
(ChatGroq, PromptTemplate, PydanticOutputParser) used by the LLM router
(Cell 3) and every AI feature that follows.
"""

from enum import Enum
import re
import uuid
import json
from datetime import datetime, timezone
from typing import Optional, List, Dict, Any, Callable, Union, Literal
import unicodedata
from functools import lru_cache

from pydantic import BaseModel, ConfigDict, Field, HttpUrl
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
import os, re, sys, time, hashlib, requests
from collections import deque
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pypdf import PdfReader
import docx2txt

In [ ]:
"""
Cell 3 — Core data models

Defines every Pydantic schema shared across features:
  - CandidateProfile family (Experience, Education, Project, Portfolio)
  - CV analysis output (CVSections, CVAnalysisResult)
  - Portfolio analysis output (PortfolioSections, ProjectFeedback, PortfolioAnalysisResult)
  - Job description output (JDAnalysisResult, JDMatchResult)
  - Interview engine enums, models, prompts, and the CreateInterview /
    SubmitAnswer / EndInterview functions.

NOTE: the interview engine functions defined at the bottom of this cell
(invoke_with_schema, CreateInterview, SubmitAnswer, EndInterview) are a
first draft. Cell 8 redefines all of them with the current, polished
docstrings — since Python re-executes definitions top-to-bottom, Cell 8's
versions are what the notebook actually runs with. This cell is still the
single source of truth for every model/enum/prompt used by the engine.
"""

class Experience(BaseModel):
    """One entry in a candidate's work-history list."""
    job_title: str
    company: str
    duration_months: Optional[int]
    description: Optional[str]


class Education(BaseModel):
    """One entry in a candidate's education history."""
    degree: str
    field_of_study: str
    institution: str
    graduation_year: Optional[int]


class Project(BaseModel):
    """One entry in a candidate's project list (CV or profile)."""
    name: str
    description: Optional[str]
    tech_used: Optional[List[str]]
    url: Optional[str]


class Portfolio(BaseModel):
    """A single external portfolio link and its recognized type."""
    url: HttpUrl
    type: Literal["github", "other"]


class CandidateProfile(BaseModel):
    """
    Full structured candidate profile — the output of ExtractCandidateProfile
    (Cell 4) and the shape a Candidate's Professional Profile is stored in.
    """
    # ── Identity ──────────────────────────────────────────
    full_name: str = Field(description="Candidate's full name")
    email: Optional[str] = Field(default=None)
    phone: Optional[str] = Field(default=None)
    location: Optional[str] = Field(default=None, description="City, Country")
    linkedin_url: Optional[str] = Field(default=None)
    github_url: Optional[str] = Field(default=None)
    portfolio_url: Optional[str] = Field(default=None)

    # ── Professional Summary ───────────────────────────────
    headline: Optional[str] = Field(default=None, description="One-line professional title or tagline")
    summary: Optional[str] = Field(default=None, description="Short bio or about section")
    years_of_experience: Optional[float] = Field(default=None, description="Total years of professional experience")

    # ── Role Targeting ─────────────────────────────────────
    desired_role: Optional[str] = Field(default=None, description="Job title the candidate is targeting")
    employment_type: Optional[Literal["full-time", "part-time", "freelance", "internship", "contract"]] = None
    work_mode: Optional[Literal["remote", "on-site", "hybrid"]] = None
    open_to_relocation: Optional[bool] = None
    expected_salary: Optional[str] = None

    # ── Skills ─────────────────────────────────────────────
    technical_skills: Optional[List[str]] = Field(default=None, description="Programming languages, tools, frameworks")
    soft_skills: Optional[List[str]] = Field(default=None, description="Communication, leadership, etc.")
    languages_spoken: Optional[List[str]] = None

    # ── Experience ─────────────────────────────────────────
    experience: Optional[List[Experience]] = None
    current_company: Optional[str] = None
    current_job_title: Optional[str] = None
    is_currently_employed: Optional[bool] = None

    # ── Education ──────────────────────────────────────────
    education: Optional[List[Education]] = None
    highest_degree: Optional[Literal["high school", "bachelor", "master", "phd", "bootcamp", "self-taught"]] = None

    # ── Projects & Portfolio ───────────────────────────────
    projects: Optional[List[Project]] = None
    portfolio_links: Optional[List[Portfolio]] = None

    # ── Certifications ─────────────────────────────────────
    certifications: Optional[List[str]] = Field(default=None, description="e.g. AWS ML/AI, Google Cloud, etc.")

    # ── Metadata ───────────────────────────────────────────
    availability: Optional[str] = Field(default=None, description="e.g. Immediate, 2 weeks notice, 1 month")


class ProfileEvaluationResult(BaseModel):
    """Full output schema of ProfileEvaluation — the Profile Completeness/Quality feature."""
    completenessScore: int = Field(description="0-100, how complete the profile is (key fields filled)")
    qualityScore: int = Field(description="0-100, quality/clarity of what IS filled in — specificity of headline/summary/skills, impact in experience")
    missingFields: List[str] = Field(description="Important profile fields that are empty and should be filled, by field name")
    weakFields: List[str] = Field(description="Fields that are filled but vague, too short, or low-signal, by field name")
    strengths: List[str] = Field(description="3-5 short bullet points")
    recommendations: List[str] = Field(description="3-5 short, actionable bullet points to strengthen the profile")
    summary: str = Field(description="1-2 sentence overall summary")


class CVSections(BaseModel):
    """Per-section feedback block returned inside CVAnalysisResult."""
    experience: str = Field(description="1-2 sentence feedback on work experience, be concise")
    education: str = Field(description="1-2 sentence feedback on education, be concise")
    skills: str = Field(description="1-2 sentence feedback on technical/soft skills, be concise")
    projects: str = Field(description="1-2 sentence feedback on projects, be concise")
    summary: str = Field(description="1-2 sentence feedback on the headline/summary section, or note if missing")
    achievements: str = Field(description="1-2 sentence feedback on quantified achievements and metrics")
    keywords: str = Field(description="1-2 sentence feedback on keyword density and ATS alignment")


class CVAnalysisResult(BaseModel):
    """Full output schema of ResumeEvaluation (Cell 5) — the CV Score feature."""
    score: int = Field(description="Overall resume quality score, 0-100")
    atsScore: int = Field(description="Semantic ATS match score judged by LLM, 0-100")
    atsTfidfScore: int = Field(description="Keyword overlap ATS score from TF-IDF, 0-100")
    grammarScore: int = Field(description="Grammar and language quality score, 0-100")
    formattingScore: int = Field(description="Formatting and structure clarity score, 0-100")
    sections: CVSections
    strengths: List[str] = Field(description="3-5 short bullet points")
    weaknesses: List[str] = Field(description="3-5 short bullet points")
    itemsToRemove: List[str] = Field(description="3-5 things that should be cut or shortened")
    missingSkills: List[str] = Field(description="List of missing skills, keywords only")
    recommendations: List[str] = Field(description="3-5 short, actionable bullet points")


class PortfolioSections(BaseModel):
    """Per-section feedback block returned inside PortfolioAnalysisResult."""
    design: str = Field(description="1-2 sentence feedback on visual design/UX — for non-design roles, evaluate layout clarity instead")
    content: str = Field(description="1-2 sentence feedback on writing/content quality, be concise")
    projects: str = Field(description="1-2 sentence feedback on project depth and case studies, be concise")
    professionalism: str = Field(description="1-2 sentence feedback on overall professionalism/presentation, be concise")
    mobileExperience: str = Field(description="1-2 sentence feedback on how the portfolio reads on mobile / small screens")
    recruiterExperience: str = Field(description="1-2 sentence feedback on how easy it is for a recruiter to find key info quickly")
    roleClarity: str = Field(description="1-2 sentence feedback on whether the candidate's contribution is clear in each project")


class ProjectFeedback(BaseModel):
    """One specific, per-project issue surfaced inside a portfolio review."""
    name: str = Field(description="Project name or repo name")
    issue: str = Field(description="One-line specific issue with this project")


class PortfolioAnalysisResult(BaseModel):
    """Full output schema of PortfolioEvaluation (Cell 6) — the Portfolio Analyzer feature."""
    score: int = Field(description="Overall portfolio quality score, 0-100")
    sections: PortfolioSections
    strengths: List[str] = Field(description="3-5 short bullet points")
    weaknesses: List[str] = Field(description="3-5 short bullet points")
    missingElements: List[str] = Field(description="Missing elements, keywords/phrases only, e.g. 'case studies', 'measurable results', 'GitHub link'")
    recommendations: List[str] = Field(description="3-5 short, actionable bullet points")
    projectFeedback: Optional[List[ProjectFeedback]] = Field(default=None, description="Per-project specific issues, max 5")


class JDAnalysisResult(BaseModel):
    """Full output schema of ParseJobDescription / IntakeJobDescription (Cell 7)."""
    job_title: str = Field(description="Exact job title from the JD")
    company: Optional[str] = Field(default=None, description="Company name if mentioned")
    experience_level: Optional[Literal["entry", "junior", "mid", "senior", "lead", "executive"]] = Field(
        default=None, description="Seniority level inferred from requirements"
    )
    years_of_experience: Optional[str] = Field(default=None, description="e.g. '2-4 years', '5+ years'")
    employment_type: Optional[Literal["full-time", "part-time", "contract", "internship", "freelance"]] = None
    work_mode: Optional[Literal["remote", "on-site", "hybrid"]] = None
    location: Optional[str] = Field(default=None, description="City / country if mentioned")
    salary_range: Optional[str] = Field(default=None, description="Salary range if mentioned")

    required_skills: List[str] = Field(description="Hard skills explicitly required — tools, languages, frameworks")
    preferred_skills: List[str] = Field(description="Nice-to-have skills, not strictly required")
    soft_skills: List[str] = Field(description="Soft skills mentioned, e.g. communication, teamwork")
    responsibilities: List[str] = Field(description="Key responsibilities, max 8, one line each")
    qualifications: List[str] = Field(description="Education or certification requirements")
    keywords: List[str] = Field(description="All ATS-relevant keywords from the JD, including synonyms")


class JDMatchResult(BaseModel):
    """Output schema of ComputeJDMatch (Cell 5) — the CV↔JD Match Percentage feature."""
    matchPercentage: float = Field(description="Overall CV-to-JD match score, 0-100")
    matchedSkills: List[str]
    missingSkills: List[str]
    extraSkills: List[str] = Field(description="Skills candidate has that JD doesn't ask for")


class InterviewType(str, Enum):
    """Supported interview types (MVP scope)."""
    HR = "HR"
    BEHAVIORAL = "BEHAVIORAL"
    TECHNICAL = "TECHNICAL"
    SITUATIONAL = "SITUATIONAL"


class CandidateLevel(str, Enum):
    """Seniority level used to calibrate interview questions and scoring."""
    JUNIOR = "JUNIOR"
    MID = "MID"
    SENIOR = "SENIOR"


class Difficulty(str, Enum):
    """Interview difficulty setting, chosen at CreateInterview time."""
    EASY = "EASY"
    MEDIUM = "MEDIUM"
    HARD = "HARD"


class InterviewLanguage(str, Enum):
    """Supported interview languages."""
    EN = "EN"
    AR = "AR"

    @property
    def full_name(self) -> str:
        return {"EN": "English", "AR": "Arabic"}[self.value]


class InterviewMode(str, Enum):
    """Delivery mode of the interview. VOICE/VIDEO are transcribed upstream before reaching SubmitAnswer."""
    TEXT = "TEXT"
    VOICE = "VOICE"
    VIDEO = "VIDEO"


class InterviewStatus(str, Enum):
    """Lifecycle status of an InterviewSession."""
    CREATED = "CREATED"
    IN_PROGRESS = "IN_PROGRESS"
    COMPLETED = "COMPLETED"
    ABANDONED = "ABANDONED"


INTERVIEW_LIMITS: Dict[InterviewType, Dict[str, int]] = {
    InterviewType.HR:          {"min_main": 4, "max_main": 6, "max_followups_per_main": 1, "max_total_turns": 9},
    InterviewType.BEHAVIORAL:  {"min_main": 5, "max_main": 7, "max_followups_per_main": 2, "max_total_turns": 12},
    InterviewType.TECHNICAL:   {"min_main": 6, "max_main": 9, "max_followups_per_main": 2, "max_total_turns": 15},
    InterviewType.SITUATIONAL: {"min_main": 4, "max_main": 6, "max_followups_per_main": 1, "max_total_turns": 9},
}

QUESTION_COUNT_GUIDE: Dict[InterviewType, str] = {
    t: f"{l['min_main']} to {l['max_main']} main questions" for t, l in INTERVIEW_LIMITS.items()
}


ContextLoader = Callable[[Optional[str]], Dict[str, Any]]

def _stub_loader(source: str) -> ContextLoader:
    """Factory for a placeholder loader that always reports its section as 'not_available'."""
    def _loader(entity_id: Optional[str] = None) -> Dict[str, Any]:
        return {"source": source, "status": "not_available", "id": entity_id}
    return _loader


class ContextProviders(BaseModel):
    """
    DATA CONTRACT:
      Each loader takes an id (candidate_id or job_id) and returns a dict.
      Until the real module is ready, its stub marks that section of the
      interview context as 'not_available' instead of failing the flow.
    """
    model_config = ConfigDict(arbitrary_types_allowed=True)

    load_candidate: ContextLoader = Field(default_factory=lambda: _stub_loader("candidate_profile"))
    load_job: ContextLoader = Field(default_factory=lambda: _stub_loader("job"))
    load_resume: ContextLoader = Field(default_factory=lambda: _stub_loader("resume"))
    load_portfolio: ContextLoader = Field(default_factory=lambda: _stub_loader("portfolio"))
    load_skills: ContextLoader = Field(default_factory=lambda: _stub_loader("skills"))


DEFAULT_PROVIDERS = ContextProviders()


class QuestionFeedback(BaseModel):
    """Per-answer feedback, generated after every SubmitAnswer call."""
    score: int = Field(description="0-100")
    strengths: List[str] = Field(description="1-3 short bullet points")
    weaknesses: List[str] = Field(description="1-3 short bullet points")
    better_answer: str = Field(description="A concise example of a stronger answer")
    improvement_advice: str = Field(description="1-2 sentence actionable advice")


class InterviewTurn(BaseModel):
    """One question/answer/feedback triple in an interview transcript."""
    question: str
    answer: Optional[str] = None
    is_followup: bool = False
    feedback: Optional[QuestionFeedback] = None


class InitialQuestion(BaseModel):
    """LLM output schema for the opening interview question."""
    question: str = Field(description="The first interview question, written in the interview language")


class NextTurnDecision(BaseModel):
    """LLM output schema deciding the next turn: follow-up, new question, or end."""
    should_end: bool = Field(description="True if the interview has covered enough ground and should end")
    is_followup: bool = Field(description="True if this question follows up on the candidate's last answer")
    question: Optional[str] = Field(default=None, description="The next question to ask. Null if should_end is true")


class InterviewEvaluation(BaseModel):
    """Holistic end-of-interview scorecard, produced by EndInterview."""
    communication: int = Field(description="0-100")
    confidence: int = Field(description="0-100")
    technical_knowledge: int = Field(description="0-100")
    problem_solving: int = Field(description="0-100")
    professionalism: int = Field(description="0-100")
    critical_thinking: int = Field(description="0-100")
    domain_knowledge: int = Field(description="0-100")
    language_proficiency: int = Field(description="0-100, proficiency in the interview language")
    answer_quality: int = Field(description="0-100")
    overall_score: int = Field(description="0-100, weighted overall performance")
    summary: str = Field(description="2-3 sentence overall summary")
    strengths: List[str] = Field(description="3-5 short bullet points")
    weaknesses: List[str] = Field(description="3-5 short bullet points")
    recommendations: List[str] = Field(description="3-5 short, actionable bullet points")


class InterviewReport(BaseModel):
    """Persisted, shareable Interview Report — combines the evaluation with the full transcript."""
    interview_id: str
    candidate_id: str
    target_role: Optional[str]
    interview_type: InterviewType
    level: CandidateLevel
    difficulty: Difficulty
    language: InterviewLanguage
    total_questions: int
    total_answered: int
    evaluation: InterviewEvaluation
    per_question: List[InterviewTurn]
    generated_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


class InterviewSession(BaseModel):
    """Mutable state object threaded through CreateInterview → SubmitAnswer → EndInterview."""
    interview_id: str
    candidate_id: str
    job_id: Optional[str] = None
    target_role: Optional[str] = None

    interview_type: InterviewType
    level: CandidateLevel
    difficulty: Difficulty
    language: InterviewLanguage = InterviewLanguage.EN
    mode: InterviewMode = InterviewMode.TEXT

    status: InterviewStatus = InterviewStatus.CREATED
    created_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

    context: Dict[str, Any] = Field(default_factory=dict)

    turns: List[InterviewTurn] = Field(default_factory=list)
    current_question_index: int = -1

    evaluation: Optional[InterviewEvaluation] = None
    report: Optional[InterviewReport] = None

    @property
    def questions(self) -> List[str]:
        return [t.question for t in self.turns]

    @property
    def answers(self) -> List[Optional[str]]:
        return [t.answer for t in self.turns]


class SkillPriority(BaseModel):
    """One missing or weak skill, ranked for the candidate's action plan."""
    skill: str
    status: Literal["missing", "weak"] = Field(
        description="Whether the skill is absent entirely or present but underdeveloped"
    )
    priority: Literal["critical", "high", "medium", "low"] = Field(
        description="How urgently this should be closed for the target role"
    )
    reason: str = Field(
        description="1 sentence: why this matters for the target role, grounded in the signals provided"
    )



class SkillGapAnalysisResult(BaseModel):
    """Full output schema of AnalyzeSkillGap — the Skill Gap Engine."""
    target_role: Optional[str] = Field(default=None, description="Role the gap was evaluated against")
    currentSkills: List[str] = Field(description="Skills the candidate currently demonstrates, deduplicated/lowercased")
    requiredSkills: List[str] = Field(description="Skills required or preferred for the target role")
    matchedSkills: List[str] = Field(description="Required skills the candidate already has")
    missingSkills: List[str] = Field(description="Required skills with no evidence in any input signal")
    weakSkills: List[str] = Field(
        description="Skills the candidate has some evidence of, but that look shallow or underdeveloped "
                     "given the resume, portfolio, or interview signals"
    )
    priority: List[SkillPriority] = Field(
        description="missingSkills + weakSkills ranked by priority, most critical first"
    )
    readinessScore: int = Field(description="0-100 overall readiness for the target role given current vs. required skills")
    summary: str = Field(description="2-3 sentence summary of the candidate's fit and biggest gap")
    recommendations: List[str] = Field(description="3-5 short, actionable steps to close the highest-priority gaps")



class DimensionWeight(BaseModel):
    """The LLM's judgment of how much ONE available dimension matters for THIS target role."""
    dimension: str = Field(description="Must be exactly one of the dimension keys listed as available — no others")
    weight: float = Field(description="Relative importance for this specific target role, 0-1")
    rationale: str = Field(description="Why the model weighted this dimension the way it did, for this role")
    evidence: List[str] = Field(default_factory=list, description="Short source-tagged facts behind this dimension's score")


class ReadinessDimension(BaseModel):
    """One readiness dimension actually backed by evidence — dimensions with no data simply don't appear."""
    name: str
    score: int = Field(description="0-100")
    weight: float = Field(description="Final normalized weight applied in the overall score, sums to 1.0 across all dimensions")
    rationale: str = Field(description="Why the model weighted this dimension the way it did, for this role")


class CriticalGap(BaseModel):
    """One critical gap or weakness blocking career readiness."""
    dimension: str = Field(description="Which readiness dimension this gap belongs to")
    severity: Literal["critical", "high", "medium"]
    description: str = Field(description="1-2 sentences, specific and grounded in the evidence given")
    impact: str = Field(description="1 sentence on why this matters for the target role")


class NextAction(BaseModel):
    """One concrete step in the candidate's improvement plan."""
    action: str = Field(description="1 concrete, specific next step")
    priority: Literal["immediate", "short_term", "long_term"]
    rationale: str = Field(description="1 short sentence tying the action back to a gap or dimension")


class CareerReadinessNarrative(BaseModel):
    """Full LLM-generated judgment (steps 4, 6, 7, 9) — weighting AND everything qualitative."""
    dimensionWeights: List[DimensionWeight] = Field(
        description="Exactly one entry per dimension listed as available in the prompt — nothing more, "
                     "nothing less. Weights must reflect genuine relative importance for THIS target role "
                     "and should sum to ~1.0 across those dimensions."
    )
    criticalGaps: List[CriticalGap] = Field(description="2-5 most critical gaps, most severe first")
    strengths: List[str] = Field(description="3-5 short bullet points, grounded in the evidence given")
    recommendations: List[str] = Field(description="3-5 short, actionable, prioritized bullet points")
    nextActions: List[NextAction] = Field(description="3-6 concrete steps forming an improvement plan, ordered by priority")
    summary: str = Field(description="2-4 sentence overall narrative summary of the candidate's readiness")


class CareerReadinessResult(BaseModel):
    """Full output schema of EvaluateCareerReadiness — the Career Readiness Orchestrator."""
    target_role: Optional[str] = None
    readiness_mode: Literal["general", "job_specific"]
    overallReadinessScore: int = Field(description="0-100, weighted across all available dimensions")
    readinessLevel: Literal["not_ready", "emerging", "developing", "ready", "highly_ready"]
    dimensions: List[ReadinessDimension] = Field(description="Only dimensions actually backed by evidence")
    dataCompleteness: Dict[str, bool]
    criticalGaps: List[CriticalGap]
    strengths: List[str]
    recommendations: List[str]
    nextActions: List[NextAction]
    summary: str

In [ ]:
"""
Cell 4 — LLM router

Wires up two Groq-hosted models — a small primary model for speed/cost, and
a larger fallback for resilience — behind a single `llm` object that every
AI feature in the notebook (CV extraction/evaluation, portfolio evaluation,
JD parsing, interview engine) calls through.
"""
llm_primary = ChatGroq(model='openai/gpt-oss-20b', temperature=0, max_tokens=8192)
llm_fallback = ChatGroq(model='openai/gpt-oss-120b', temperature=0, max_tokens=8192)

class LLMRouter:
    """
    Thin routing wrapper around two LangChain chat models.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        primary: LangChain chat model, tried first on every invoke().
        fallback: LangChain chat model, used only if primary raises.
      behavior:
        invoke(prompt_str) calls primary.invoke(...); on any exception it
        logs the failure and retries once against fallback.invoke(...).
      status: active
    """
    def __init__(self, primary, fallback):
        self.primary = primary
        self.fallback = fallback

    def invoke(self, prompt_str: str):
        try:
            return self.primary.invoke(prompt_str)
        except Exception as e:
            print(f"[LLMRouter] Primary model failed ({e}), falling back...")
            return self.fallback.invoke(prompt_str)

llm = LLMRouter(llm_primary, llm_fallback)

def invoke_with_schema(prompt_str: str, parser: "PydanticOutputParser", max_retries: int = 2):
    """
    Calls the LLM router and retries with the validation error appended to
    the prompt if the response fails to parse against `parser`.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        prompt_str (str): Fully-formatted prompt.
        parser (PydanticOutputParser): Schema the LLM output must validate against.
        max_retries (int, default=2): Extra retry attempts on parse failure.
      output_schema: An instance of the Pydantic model `parser` was built with.
      status: active
      error_schema:
        RuntimeError: Raised after max_retries+1 failed parse attempts.
    """
    last_error = None
    for attempt in range(1, max_retries + 2):
        raw_output = llm.invoke(prompt_str)
        try:
            return parser.parse(raw_output.content)
        except Exception as e:
            last_error = e
            print(f"[invoke_with_schema] Parse attempt {attempt} failed: {e}")
            prompt_str = (
                f"{prompt_str}\n\n"
                f"Your previous response failed validation with this error:\n{e}\n\n"
                f"Your previous (invalid) response was:\n{raw_output.content}\n\n"
                f"Return ONLY a corrected, complete JSON object matching the schema exactly."
            )
    raise RuntimeError(f"invoke_with_schema failed after {max_retries + 1} attempts. Last error: {last_error}")

In [ ]:
"""
Cell 5 — Feature: Candidate Profile extraction

Turns raw text (CV content, onboarding form, or a free-form bio) into a
structured CandidateProfile, powering the "build your Professional Profile"
onboarding step.
"""
output_parser = PydanticOutputParser(pydantic_object=CandidateProfile)
profile_prompt = PromptTemplate(
    template=(
        """You are an expert at extracting candidate profile information.
Extract all available fields from the text below.
If a field is not mentioned, leave it as null.

{format_instructions}

Candidate profile:
{input}"""
    ),
    input_variables=["input"],
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    }
)

extraction_chain = profile_prompt | llm_primary | output_parser

def ExtractCandidateProfile(text: str, max_retries: int = 2) -> dict:
    """
    Extracts structured candidate profile from raw text.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        text (str): Raw candidate text — CV content, onboarding form, or free-form bio.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema: CandidateProfile — full_name, email, phone, location, linkedin_url,
        github_url, portfolio_url, headline, summary, years_of_experience, desired_role,
        employment_type, work_mode, open_to_relocation, expected_salary, technical_skills,
        soft_skills, languages_spoken, experience, education, projects, portfolio_links,
        certifications, availability.
      status: active
      error_schema:
        RuntimeError: Raised after max_retries+1 failed parse attempts.
      example_request:
        ExtractCandidateProfile("John Doe — ML Engineer. Skills: Python, PyTorch...")
      example_response:
        { "full_name": "John Doe", "technical_skills": ["Python", "PyTorch"], ... }
    """
    last_error = None
    prompt_str = profile_prompt.format(input=text)
    for attempt in range(1, max_retries + 2):
        raw_output = llm.invoke(prompt_str)
        try:
            result: CandidateProfile = output_parser.parse(raw_output.content)
            return result.model_dump_json(indent=2)
        except Exception as e:
            last_error = e
            print(f"[ExtractCandidateProfile] Parse attempt {attempt} failed: {e}")
            prompt_str = (
                f"{profile_prompt.format(input=text)}\n\n"
                f"Your previous response failed validation with this error:\n{e}\n\n"
                f"Your previous (invalid) response was:\n{raw_output.content}\n\n"
                f"Return ONLY a corrected, complete JSON object. Include every required field."
            )
    raise RuntimeError(f"ExtractCandidateProfile failed after {max_retries + 1} attempts. Last error: {last_error}")


PROFILE_IMPORTANT_FIELDS = [
    "full_name", "email", "location", "headline", "summary",
    "years_of_experience", "desired_role", "technical_skills",
    "experience", "education", "projects", "portfolio_links",
]

profile_evaluation_parser = PydanticOutputParser(pydantic_object=ProfileEvaluationResult)

profile_evaluation_prompt = PromptTemplate(
    template=(
        """You are an expert recruiter reviewing a candidate's professional profile for
completeness and quality, ahead of it being shown to recruiters or used in job matching.

Candidate profile (JSON):
{profile_json}

Deterministic completeness reference (treat as a starting point, not ground truth —
correct or extend it using your own read of the profile below):
{completeness_pct}% of key fields filled. Empty key fields detected: {missing_fields_ref}

Evaluate:
- completenessScore: overall completeness (0-100), informed by the reference above but
  using your own judgment of what actually matters for this candidate.
- qualityScore: quality of what IS filled in — is the headline/summary specific and
  compelling rather than generic, are skills concrete, is experience described with impact.
- missingFields: important fields that are empty and should be filled (use field names,
  e.g. "headline", "summary", "portfolio_links").
- weakFields: fields that are filled but vague, too short, or low-signal.
- Do not penalize fields that are genuinely not applicable (e.g. certifications for a
  junior candidate with none yet).

{format_instructions}
"""
    ),
    input_variables=["profile_json", "completeness_pct", "missing_fields_ref"],
    partial_variables={"format_instructions": profile_evaluation_parser.get_format_instructions()},
)


def _compute_completeness_ref(candidate: dict) -> tuple:
    """Deterministic % of PROFILE_IMPORTANT_FIELDS that are filled, + which ones are empty."""
    missing = [f for f in PROFILE_IMPORTANT_FIELDS if not candidate.get(f)]
    filled = len(PROFILE_IMPORTANT_FIELDS) - len(missing)
    pct = round(filled / len(PROFILE_IMPORTANT_FIELDS) * 100)
    return pct, missing


def ProfileEvaluation(candidate: dict, max_retries: int = 2) -> dict:
    """
    Evaluates a candidate's structured profile for completeness and quality.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        candidate (dict): CandidateProfile dict — output of ExtractCandidateProfile,
          or the candidate's current stored profile.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema: ProfileEvaluationResult — completenessScore, qualityScore,
        missingFields, weakFields, strengths, recommendations, summary.
      status: active
      error_schema:
        RuntimeError: Raised after max_retries+1 failed parse attempts.
      example_request:
        ProfileEvaluation(extracted_profile)
      example_response:
        { "completenessScore": 67, "qualityScore": 58,
          "missingFields": ["headline", "portfolio_links"], "weakFields": ["summary"], ... }
    """
    completeness_pct, missing_fields_ref = _compute_completeness_ref(candidate)

    prompt_str = profile_evaluation_prompt.format(
        profile_json=json.dumps(candidate, default=str, indent=2),
        completeness_pct=completeness_pct,
        missing_fields_ref=missing_fields_ref or "none",
    )

    result: ProfileEvaluationResult = invoke_with_schema(prompt_str, profile_evaluation_parser, max_retries=max_retries)
    return result.model_dump_json(indent=2)

In [ ]:
"""
Cell 6: Skill normalization layer
-------------------------
Maps every spelling of a skill ("JS", "ReactJS", "k8s", "Postgres", "ML"...) onto ONE
canonical lowercase name, and matches skills inside free text on WORD BOUNDARIES
(so "js" no longer matches inside "json", "go" inside "google", "ml" inside "html").

Public API:
    normalize_skill("ReactJS")                 -> "react"
    normalize_skills(["JS", "javascript"])     -> ["javascript"]
    skill_in_text("k8s", "deployed on Kubernetes")  -> True
"""


SKILL_ALIASES: Dict[str, List[str]] = {
    "python": ["python3", "py3"],
    "javascript": ["js", "ecmascript", "es6"],
    "typescript": ["ts"],
    "c++": ["cpp"],
    "c#": ["csharp"],
    "go": ["golang"],
    "r": ["r language", "r programming", "rstudio"],
    "c": ["c language", "c programming"],
    ".net": ["dotnet", "dot net", "asp.net", "aspnet"],
    "artificial intelligence": ["ai"],
    "machine learning": ["ml"],
    "deep learning": ["dl"],
    "natural language processing": ["nlp"],
    "large language models": ["llm", "llms", "large language model"],
    "retrieval augmented generation": ["rag"],
    "generative ai": ["genai", "gen ai", "generative artificial intelligence"],
    "vector database": ["vector db", "vectordb", "vector store"],
    "hugging face": ["huggingface", "hf transformers"],
    "pytorch": ["torch"],
    "tensorflow": ["tf"],
    "scikit-learn": ["sklearn", "scikit learn"],
    "mlops": ["ml ops", "machine learning operations"],
    "multi-agent systems": ["multi agent", "multi-agent", "multiagent systems"],
    "react": ["reactjs", "react.js"],
    "vue": ["vuejs", "vue.js"],
    "angular": ["angularjs", "angular.js"],
    "node.js": ["nodejs", "node"],
    "next.js": ["nextjs"],
    "express": ["expressjs", "express.js"],
    "fastapi": ["fast api"],
    "rest api": ["rest", "restful", "restful api", "rest apis", "restful apis"],
    "html": ["html5"],
    "css": ["css3"],
    "sql": ["structured query language"],
    "postgresql": ["postgres", "psql"],
    "mongodb": ["mongo"],
    "sql server": ["mssql", "microsoft sql server", "ms sql"],
    "power bi": ["powerbi", "microsoft power bi"],
    "excel": ["microsoft excel", "ms excel"],
    "aws": ["amazon web services"],
    "gcp": ["google cloud", "google cloud platform"],
    "azure": ["microsoft azure"],
    "kubernetes": ["k8s"],
    "ci/cd": ["cicd", "continuous integration", "continuous delivery", "continuous deployment"],
    "object-oriented programming": ["oop", "object oriented programming", "object oriented"],
    "data structures and algorithms": ["dsa", "data structures & algorithms", "data structures", "algorithms and data structures"],
    "communication": ["communication skills"],
    "problem solving": ["problem-solving skills", "problem solving skills"],
    "teamwork": ["team work", "team player"],
}


_AMBIGUOUS_BARE = {"go", "r", "c"}

_SEP = r"[\s.\-_/]"


def _compact(s: str) -> str:
    """'React.js' / 'react js' / 'React-JS' -> 'reactjs'. Keeps + and # (c++, c#)."""
    return re.sub(_SEP, "", s.lower())


def _clean(skill: str) -> str:
    s = unicodedata.normalize("NFKC", str(skill)).lower()
    s = re.sub(r"\(.*?\)", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\s+v?\d+(?:\.\d+)*\+?$", "", s)
    return s.lstrip("•*·- ").rstrip(",;:.•* ")

_ALIAS_LOOKUP: Dict[str, str] = {}
for _canon, _aliases in SKILL_ALIASES.items():
    _ALIAS_LOOKUP[_compact(_canon)] = _canon
    for _a in _aliases:
        _ALIAS_LOOKUP[_compact(_a)] = _canon


def normalize_skill(skill: Optional[str]) -> str:
    """Returns the canonical lowercase form of a skill (or its cleaned form if unknown)."""
    if not skill:
        return ""
    s = _clean(skill)
    return _ALIAS_LOOKUP.get(_compact(s), s)


def normalize_skills(skills: Optional[List[str]]) -> List[str]:
    """Canonicalizes and dedupes a skill list, preserving first-seen order."""
    seen, out = set(), []
    for s in skills or []:
        n = normalize_skill(s)
        if n and n not in seen:
            seen.add(n)
            out.append(n)
    return out


@lru_cache(maxsize=2048)
def _skill_pattern(canon: str) -> "re.Pattern":
    forms = {canon, *SKILL_ALIASES.get(canon, [])}
    if canon in _AMBIGUOUS_BARE:
        forms.discard(canon)
    alts = []
    for f in forms:
        tokens = [re.escape(t) for t in re.split(_SEP, f) if t]
        lead = r"\." if f.startswith(".") else ""
        alts.append(lead + r"[\s.\-_/]*".join(tokens))
    body = "|".join(sorted(alts, key=len, reverse=True))
    return re.compile(rf"(?<![a-z0-9])(?<![a-z0-9]\.)(?:{body})(?![a-z0-9])")


def skill_in_text(skill: str, text: str) -> bool:
    """True if the skill (any alias/spelling) appears in `text` as a whole word."""
    canon = normalize_skill(skill)
    if not canon or not text:
        return False
    return bool(_skill_pattern(canon).search(unicodedata.normalize("NFKC", text).lower()))

In [ ]:
"""
Cell 7 — Feature: CV Analysis, ATS Scoring, and CV↔JD Match

Extracts text from an uploaded resume file (.pdf/.docx), scores it (CV Score,
ATS Score, Grammar, Formatting) with concrete section-by-section feedback,
and — when a job description is supplied — computes a deterministic
keyword/TF-IDF Match Percentage against it.
"""


def Resume_analysis(file_path):
    """Extracts raw text from a .pdf or .docx resume file (pypdf / docx2txt, no LangChain dependency)."""
    path = file_path.lower()
    if path.endswith(".pdf"):
        reader = PdfReader(file_path)
        return "\n".join((page.extract_text() or "") for page in reader.pages)
    elif path.endswith(".docx"):
        return docx2txt.process(file_path)
    else:
        raise ValueError("Unsupported file type. Only .pdf and .docx are supported.")


resume_evaluation_parser = PydanticOutputParser(pydantic_object=CVAnalysisResult)

resume_evaluation_prompt = PromptTemplate(
      template=(
"""You are an expert technical recruiter and ATS system.
Evaluate the candidate's resume against the target job description.

IMPORTANT: Keep every field CONCISE.
- atsScore: your semantic judgment of ATS fit (0-100), independent of the TF-IDF score.
- atsTfidfScore: will be set externally — output 0 as a placeholder.
- grammarScore: rate grammar, spelling, and language quality (0-100).
- formattingScore: rate clarity of structure, use of whitespace, section hierarchy (0-100).
- itemsToRemove: list specific lines, sections, or content that weakens the CV and should be cut.
- sections.summary: give feedback on the headline/summary, or flag if it is missing entirely.
- sections.achievements: flag whether achievements are quantified with numbers and results.
- sections.keywords: assess keyword density and relevance to the role.
""".lstrip() +
"""
{format_instructions}

Resume:
{resume_text}

Job Description:
{job_description}

TF-IDF keyword match score (reference only): {tfidf_score}%
"""
    ),
    input_variables=["resume_text", "job_description", "tfidf_score"],
    partial_variables={"format_instructions": resume_evaluation_parser.get_format_instructions()}
)


def compute_tfidf_ats_score(resume_text: str, description: str) -> float:
    """Deterministic keyword-overlap score, used as the ATS signal + fed to the LLM as context."""
    vectorizer = TfidfVectorizer(stop_words="english")
    if not description or not description.strip():
        return 0.0
    tfidf_matrix = vectorizer.fit_transform([resume_text, description])
    score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return round(score * 100, 2)


def ComputeJDMatch(resume_text: str, parsed_jd: dict) -> dict:
    """
    Computes CV-to-JD match using TF-IDF cosine similarity + keyword overlap.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        resume_text (str): Plain extracted text of the candidate's resume.
        parsed_jd (dict): Output of ParseJobDescription / IntakeJobDescription.
      output_schema: JDMatchResult — matchPercentage (float 0-100), matchedSkills (list[str]),
        missingSkills (list[str]), extraSkills (list[str], currently always []).
      status: active
      error_schema: No exceptions raised. Returns 0% scores if JD has no skills listed.
      example_request:
        ComputeJDMatch("Experienced in Python, PyTorch...", parsed_jd)
      example_response:
        { "matchPercentage": 68.5, "matchedSkills": ["python", "pytorch"], "missingSkills": ["go"], "extraSkills": [] }
    """
    all_jd_skills = set(normalize_skills(
        list(parsed_jd.get("required_skills", [])) + list(parsed_jd.get("preferred_skills", []))
    ))
    matched = {s for s in all_jd_skills if skill_in_text(s, resume_text)}
    missing = all_jd_skills - matched
    tfidf = compute_tfidf_ats_score(resume_text, " ".join(all_jd_skills)) if all_jd_skills else 0.0
    skill_pct = (len(matched) / len(all_jd_skills) * 100) if all_jd_skills else 0.0
    return JDMatchResult(
        matchPercentage=round((tfidf + skill_pct) / 2, 2),
        matchedSkills=sorted(matched),
        missingSkills=sorted(missing),
        extraSkills=[],
    ).model_dump()


def ResumeEvaluation(file_path: str, description: Union[str, dict, None] = None, max_retries: int = 2) -> dict:
    """
    Evaluates a resume file against an optional job description.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        file_path (str): Absolute path to a .pdf or .docx resume file.
        description (str | dict | None): Raw JD text, pre-parsed JD dict, or None for general eval.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema: CVAnalysisResult — score, atsScore, atsTfidfScore, grammarScore,
        formattingScore, sections (CVSections), strengths, weaknesses, itemsToRemove,
        missingSkills, recommendations.
      status: active
      error_schema:
        ValueError: Unsupported file type (only .pdf and .docx accepted).
        RuntimeError: Raised after max_retries+1 failed parse attempts.
      example_request:
        ResumeEvaluation("/files/cv.pdf", "We are hiring a senior ML engineer...")
      example_response:
        { "score": 78, "atsScore": 82, "atsTfidfScore": 71, "strengths": [...], ... }
    """
    resume_text = Resume_analysis(file_path)

    jd_provided = bool(description) if isinstance(description, dict) else bool(description and description.strip())

    if jd_provided:
      parsed_jd = description if isinstance(description, dict) else json.loads(IntakeJobDescription(description))
      job_description_for_prompt = (
            f"Title: {parsed_jd.get('job_title')}\n"
            f"Required skills: {', '.join(parsed_jd.get('required_skills', []))}\n"
            f"Preferred skills: {', '.join(parsed_jd.get('preferred_skills', []))}\n"
            f"Responsibilities: {chr(10).join(parsed_jd.get('responsibilities', []))}\n"
            f"Keywords: {', '.join(parsed_jd.get('keywords', []))}\n"
            f"Qualifications: {', '.join(parsed_jd.get('qualifications', []))}"
      )
      jd_text_for_tfidf = (
          " ".join([
              parsed_jd.get("job_title") or "",
              *parsed_jd.get("required_skills", []),
              *parsed_jd.get("preferred_skills", []),
              *parsed_jd.get("keywords", []),
              *parsed_jd.get("responsibilities", []),
          ])
          if isinstance(description, dict) else description
      )
      tfidf_score = compute_tfidf_ats_score(resume_text, jd_text_for_tfidf)
    else:
        parsed_jd = None
        job_description_for_prompt = (
            "No specific job description was provided. "
            "Please evaluate the resume's general ATS compatibility and overall quality. "
            "Consider common ATS criteria such as clear formatting, keyword density (for general roles), "
            "action verbs, and quantifiable achievements. "
            "The TF-IDF score below is 0% as no target job description was given and is not relevant for this general assessment."
        )
        tfidf_score = 0.0

    prompt_value = resume_evaluation_prompt.format_prompt(
        resume_text=resume_text,
        job_description=job_description_for_prompt,
        tfidf_score=tfidf_score
    )
    prompt_str = prompt_value.to_string()

    last_error = None
    for attempt in range(1, max_retries + 2):
        raw_output = llm.invoke(prompt_str)
        try:
            result: CVAnalysisResult = resume_evaluation_parser.parse(raw_output.content)
            output = result.model_dump()
            output["atsTfidfScore"] = round(tfidf_score)
            return json.dumps(output, ensure_ascii=False, indent=2, default=str)
        except Exception as e:
            last_error = e
            print(f"[ResumeEvaluation] Parse attempt {attempt} failed: {e}")
            prompt_str = (
                f"{prompt_value.to_string()}\n\n"
                f"Your previous response failed validation with this error:\n{e}\n\n"
                f"Your previous (invalid) response was:\n{raw_output.content}\n\n"
                f"Return ONLY a corrected, complete JSON object matching the schema exactly. "
                f"Include every required field."
            )

    raise RuntimeError(f"ResumeEvaluation failed after {max_retries + 1} attempts. Last error: {last_error}")

In [ ]:
"""
Cell 8 — Feature: Portfolio Analysis

Scrapes a candidate's portfolio URL (GitHub via API, or a generic site via
crawler) into plain text, then evaluates design, content, project depth,
professionalism, and recruiter/mobile experience against a target role.
Behance/Dribbble are recognized but marked UNSUPPORTED_SOURCE — no reliable
automated extraction exists for them yet.
"""


STRIP_TAGS = ["script", "style", "noscript", "svg", "footer", "nav", "form", "iframe"]
same_domain = lambda url, base: urlparse(url).netloc.replace("www.", "") == base.replace("www.", "")

def clean_text(soup):
    """Strips boilerplate tags and collapses whitespace, returning readable page text."""
    for t in soup(STRIP_TAGS): t.decompose()
    lines = [l.strip() for l in soup.get_text("\n").splitlines() if l.strip()]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()

def get_links(soup, base_url, base_domain):
    """Collects same-domain page links from a parsed page, for the generic-site crawler."""
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("mailto:", "tel:", "javascript:", "#")): continue
        full = urljoin(base_url, href).split("#")[0].rstrip("/")
        if same_domain(full, base_domain): links.add(full)
    return links

def chunk_text(text, limit=300):
    """Splits text into ~`limit`-word chunks for downstream LLM context building."""
    words = text.split()
    return [" ".join(words[i:i+limit]) for i in range(0, len(words), limit)]

def dedup_chunks(chunks):
    """Removes duplicate chunks (by hash), e.g. repeated nav/footer text across pages."""
    seen, unique = set(), []
    for c in chunks:
        h = hashlib.md5(c.encode()).hexdigest()
        if h not in seen:
            seen.add(h); unique.append(c)
    return unique

def build_output(pages):
    """Formats scraped (url, text) pairs into a single labeled text blob for the LLM prompt."""
    lines = []
    for url, text in pages:
        for i, chunk in enumerate(dedup_chunks(chunk_text(text)), 1):
            lines += [f"--- SOURCE: {url} | CHUNK {i} ---", chunk, ""]
    return "\n".join(lines)


class PortfolioAdapter:
    """Base interface every portfolio-source adapter implements."""
    def supports(self, url: str) -> bool:
        raise NotImplementedError

    def extract(self, url: str, max_pages: int = 50) -> dict:
        """Must return {status, platform, content}."""
        raise NotImplementedError


class GitHubAdapter(PortfolioAdapter):
    """Extracts profile + non-fork repo metadata via the GitHub REST API (no scraping)."""
    def supports(self, url):
        return urlparse(url).netloc.replace("www.", "") == "github.com"

    def extract(self, url, max_pages=50):
        username = urlparse(url).path.strip("/").split("/")[0]
        headers = {"User-Agent": "Mozilla/5.0 (LLM-Scraper/1.0)"}
        results = []

        u_resp = requests.get(f"https://api.github.com/users/{username}", headers=headers, timeout=10)
        if u_resp.status_code == 200:
            u = u_resp.json()
            results.append((url, "\n".join(filter(None, [
                f"Name: {u.get('name')}", f"Bio: {u.get('bio')}", f"Location: {u.get('location')}",
                f"Company: {u.get('company')}", f"Public repos: {u.get('public_repos')}",
                f"Followers: {u.get('followers')}", f"Blog/website: {u.get('blog')}",
            ]))))

        r_resp = requests.get(f"https://api.github.com/users/{username}/repos?sort=updated&per_page=100", headers=headers, timeout=10)
        if r_resp.status_code == 200:
            for repo in r_resp.json():
                if repo.get("fork"): continue
                results.append((repo.get("html_url"), "\n".join(filter(None, [
                    f"Repo: {repo.get('name')}", f"Description: {repo.get('description')}",
                    f"Language: {repo.get('language')}", f"Stars: {repo.get('stargazers_count')}",
                    f"Topics: {', '.join(repo.get('topics', []))}", f"URL: {repo.get('html_url')}",
                ]))))
                sys.stdout.write(f"\rFetching GitHub repos... {len(results)} items")
                sys.stdout.flush()
        print()

        return {"status": "OK", "platform": "github", "content": build_output(results)}


class UnsupportedPlatformAdapter(PortfolioAdapter):
    """Behance, Dribbble, and anything else with no reliable automated extraction yet."""
    PLATFORMS = {"behance.net", "dribbble.com"}

    def supports(self, url):
        return urlparse(url).netloc.replace("www.", "") in self.PLATFORMS

    def extract(self, url, max_pages=50):
        return {"status": "UNSUPPORTED_SOURCE", "platform": urlparse(url).netloc.replace("www.", ""), "content": None}


class GenericWebsiteAdapter(PortfolioAdapter):
    """Fallback for personal sites, Vercel deploys, etc. Must stay last in the registry."""
    def supports(self, url):
        return True

    def extract(self, url, max_pages=50):
        base_domain = urlparse(url).netloc
        visited, queue, results = set(), deque([url]), []
        session = requests.Session()
        session.headers.update({"User-Agent": "Mozilla/5.0 (LLM-Scraper/1.0)"})

        while queue and len(visited) < max_pages:
            current = queue.popleft()
            if current in visited: continue
            visited.add(current)
            try:
                resp = session.get(current, timeout=10)
                resp.raise_for_status()
            except requests.RequestException:
                continue
            if "text/html" not in resp.headers.get("Content-Type", ""): continue

            soup = BeautifulSoup(resp.text, "html.parser")
            text = clean_text(soup)
            if text: results.append((current, text))

            for link in get_links(soup, current, base_domain):
                if link not in visited: queue.append(link)

            sys.stdout.write(f"\rCrawling... {len(visited)}/{max_pages} pages, {len(results)} with content")
            sys.stdout.flush()
            time.sleep(0.5)
        print()

        status = "PARTIAL" if (not results or len(visited) >= max_pages) else "OK"
        return {"status": status, "platform": "generic", "content": build_output(results)}


PORTFOLIO_ADAPTERS = [GitHubAdapter(), UnsupportedPlatformAdapter(), GenericWebsiteAdapter()]

def scrape_to_context(url, max_pages=50):
    """
    Route a portfolio URL to the correct adapter and return structured result.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        url (str): Portfolio URL — GitHub profile, generic website, or known platform.
        max_pages (int, default=50): Max pages to crawl for generic websites.
      output_schema:
        {
          "status": "OK" | "PARTIAL" | "UNSUPPORTED_SOURCE",
          "platform": "github" | "generic" | "unsupported",
          "url": str,
          "content": str | None
        }
      status: active
      error_schema: No exceptions raised — unsupported/partial platforms return status field.
      example_request:
        scrape_to_context("https://github.com/faresemad")
      example_response:
        { "status": "OK", "platform": "github", "url": "...", "content": "Name: Fares..." }
    """
    for adapter in PORTFOLIO_ADAPTERS:
        if adapter.supports(url):
            return adapter.extract(url, max_pages=max_pages)


portfolio_evaluation_parser = PydanticOutputParser(pydantic_object=PortfolioAnalysisResult)

portfolio_evaluation_prompt = PromptTemplate(
    template=(
        """You are an expert design/hiring reviewer evaluating a candidate's portfolio.
Evaluate the scraped portfolio content below for a candidate targeting the role: {target_role}.

Assess: design & UX, content quality, project/case-study depth, results shown, technologies used,
professionalism, and how a recruiter would experience it.

IMPORTANT: Keep every field CONCISE. Section feedback must be 1-2 sentences max.
Bullet lists must be short phrases, not full sentences. You MUST include every
field in the schema below, even if brief — do not omit any field.

{format_instructions}

Portfolio Content:
{portfolio_text}
"""
    ),
    input_variables=["portfolio_text", "target_role"],
    partial_variables={"format_instructions": portfolio_evaluation_parser.get_format_instructions()}
)


def PortfolioEvaluation(url: str, target_role: str = None, max_retries: int = 2) -> dict:
    """
    Scrapes a portfolio URL and evaluates it for the given target role.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        url (str): Full URL of the candidate's portfolio, GitHub, or generic website.
        target_role (str | None): Job title the candidate is targeting.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema:
        On success — PortfolioAnalysisResult: score, sections (PortfolioSections),
          strengths, weaknesses, missingElements, recommendations, projectFeedback.
        On unsupported — { "status": "UNSUPPORTED_SOURCE", "platform": str, "url": str, "score": null }
        On partial — PortfolioAnalysisResult with "status": "PARTIAL" added.
      status: active
      error_schema:
        RuntimeError: Raised after max_retries+1 failed LLM parse attempts.
      example_request:
        PortfolioEvaluation("https://github.com/faresemad", target_role="ML Engineer")
      example_response:
        { "status": "OK", "score": 74, "strengths": [...], "missingElements": [...], ... }
    """
    scrape_result = scrape_to_context(url)

    if scrape_result["status"] == "UNSUPPORTED_SOURCE":
      return json.dumps({
            "status": "UNSUPPORTED_SOURCE",
            "platform": scrape_result["platform"],
            "url": url,
            "score": None,
            "message": "Automated extraction is not supported for this platform. Manual review required."
        }, ensure_ascii=False, indent=2)

    portfolio_text = scrape_result["content"] or ""
    extraction_status = scrape_result["status"]

    prompt_value = portfolio_evaluation_prompt.format_prompt(
        portfolio_text=portfolio_text,
        target_role=target_role if target_role else "Not specified"
    )
    prompt_str = prompt_value.to_string()

    last_error = None
    for attempt in range(1, max_retries + 2):
        raw_output = llm.invoke(prompt_str)
        try:
            result: PortfolioAnalysisResult = portfolio_evaluation_parser.parse(raw_output.content)
            output = result.model_dump()
            output["status"] = extraction_status
            return json.dumps(output, ensure_ascii=False, indent=2, default=str)
        except Exception as e:
            last_error = e
            print(f"[PortfolioEvaluation] Parse attempt {attempt} failed: {e}")
            prompt_str = (
                f"{prompt_value.to_string()}\n\n"
                f"Your previous response failed validation with this error:\n{e}\n\n"
                f"Your previous (invalid) response was:\n{raw_output.content}\n\n"
                f"Return ONLY a corrected, complete JSON object matching the schema exactly. "
                f"Include every required field."
            )

    raise RuntimeError(f"PortfolioEvaluation failed after {max_retries + 1} attempts. Last error: {last_error}")

In [ ]:
"""
Cell 9 — Feature: Job Description parsing

Turns a raw job posting into structured fields (required/preferred skills,
responsibilities, keywords, etc.) used by ResumeEvaluation's JD comparison
and by ComputeJDMatch's match-percentage calculation.
"""
jd_parser = PydanticOutputParser(pydantic_object=JDAnalysisResult)

jd_prompt = PromptTemplate(
    template="""You are an expert recruiter and ATS specialist.
Extract all structured information from the job description below.

Rules:
- required_skills: only what is explicitly stated as required or mandatory.
- preferred_skills: anything listed as "nice to have", "bonus", "plus", or "preferred".
- keywords: every skill, tool, technology, methodology, and domain term that an ATS would scan for — include abbreviations and full forms (e.g. both "ML" and "machine learning").
- responsibilities: max 8 items, one line each, start with a verb.
- If a field is not mentioned, return an empty list or null — do not invent information.

{format_instructions}

Job Description:
{jd_text}
""",
    input_variables=["jd_text"],
    partial_variables={"format_instructions": jd_parser.get_format_instructions()}
)


def ParseJobDescription(jd_text: str, max_retries: int = 2) -> dict:
    """
    Parses a raw job description string into structured fields.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        jd_text (str): Raw job description text.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema: JDAnalysisResult — job_title, company, experience_level,
        years_of_experience, employment_type, work_mode, location, salary_range,
        required_skills, preferred_skills, soft_skills, responsibilities,
        qualifications, keywords.
      status: active
      error_schema:
        RuntimeError: Raised after max_retries+1 failed parse attempts.
      example_request:
        ParseJobDescription("We are looking for a Senior ML Engineer with 5+ years...")
      example_response:
        { "job_title": "Senior ML Engineer", "required_skills": ["Python", "PyTorch"], ... }
    """
    prompt_str = jd_prompt.format(jd_text=jd_text)
    last_error = None

    for attempt in range(1, max_retries + 2):
        raw_output = llm.invoke(prompt_str)
        try:
            result: JDAnalysisResult = jd_parser.parse(raw_output.content)
            return result.model_dump_json(indent=2)
        except Exception as e:
            last_error = e
            print(f"[ParseJobDescription] Parse attempt {attempt} failed: {e}")
            prompt_str = (
                f"{jd_prompt.format(jd_text=jd_text)}\n\n"
                f"Your previous response failed validation:\n{e}\n\n"
                f"Previous (invalid) response:\n{raw_output.content}\n\n"
                f"Return ONLY a corrected, complete JSON object. Every required field must be present."
            )

    raise RuntimeError(f"ParseJobDescription failed after {max_retries + 1} attempts. Last error: {last_error}")


def IntakeJobDescription(jd_text: str) -> dict:
    """
    Standalone intake step — call this once when a JD is submitted.
    Downstream tools (ResumeEvaluation, ComputeJDMatch) accept the returned dict directly.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        jd_text (str): Raw job description text to be parsed once and reused downstream.
      output_schema: JDAnalysisResult dict (identical to ParseJobDescription output).
      status: active
      error_schema:
        RuntimeError: Propagated from ParseJobDescription on repeated parse failure.
      example_request:
        IntakeJobDescription("Hiring a Backend Engineer with Go and Kubernetes experience...")
      example_response:
        { "job_title": "Backend Engineer", "required_skills": ["Go", "Kubernetes"], ... }
    """
    return ParseJobDescription(jd_text)

In [ ]:
"""
Cell 10 — Feature: AI Interview engine

This is the version of invoke_with_schema, CreateInterview, SubmitAnswer,
and EndInterview that the notebook actually runs with — it redefines the
v1 draft from Cell 2 with the polished CreateInterview docstring (MVP
scope note, stub-provider fallback documented). Powers the full dynamic
AI Interview loop: open with a context-aware question, score each answer
and decide the next question (follow-up or new) after every SubmitAnswer,
then produce a holistic InterviewEvaluation + InterviewReport on EndInterview.
"""


initial_question_parser = PydanticOutputParser(pydantic_object=InitialQuestion)

initial_question_prompt = PromptTemplate(
    template=(
        """You are an expert interviewer conducting a {interview_type} interview in {language}
for a {level} candidate targeting the role: {target_role}.
Calibrate the question's depth and rigor to a {difficulty} difficulty level.
The full interview should have {question_count_guide} in total — this is only the OPENING question.

Context gathered about the candidate:
{context_block}

Ask ONE strong opening question appropriate for a {interview_type} interview.
Write the question entirely in {language}.

{format_instructions}
"""
    ),
    input_variables=["interview_type", "language", "level", "difficulty", "target_role",
                      "question_count_guide", "context_block"],
    partial_variables={"format_instructions": initial_question_parser.get_format_instructions()},
)

next_turn_parser = PydanticOutputParser(pydantic_object=NextTurnDecision)

interview_next_prompt = PromptTemplate(
    template=(
        """You are an expert interviewer conducting a {interview_type} interview in {language}
for a {level} candidate targeting the role: {target_role}.
Calibrate question depth and rigor to a {difficulty} difficulty level.
The full interview should have {question_count_guide} in total (main questions, not counting follow-ups).

Context gathered about the candidate:
{context_block}

Conversation so far (question / answer pairs, in order):
{transcript}

Allowed next actions right now (hard limits are enforced by the system — choose ONLY from these):
{allowed_actions}

Decide the next action:
- If the candidate's last answer was vague, incomplete, or invites a natural follow-up, ask ONE follow-up
  question that probes deeper into that specific answer. Set is_followup to true.
- Otherwise, ask the next main question appropriate for this interview type, level and difficulty.
  Set is_followup to false.
- If enough ground has been covered for this {interview_type} interview, set should_end to true and
  leave question null.
- Never repeat a question already asked.
- Write the question entirely in {language}.

{format_instructions}
"""
    ),
    input_variables=["interview_type", "language", "level", "difficulty", "target_role",
                      "question_count_guide", "context_block", "transcript", "allowed_actions"],
    partial_variables={"format_instructions": next_turn_parser.get_format_instructions()},
)

feedback_parser = PydanticOutputParser(pydantic_object=QuestionFeedback)

question_feedback_prompt = PromptTemplate(
    template=(
        """You are an expert interview coach. Evaluate ONE answer from a {interview_type} interview,
for a {level} candidate, at {difficulty} difficulty.

Question: {question}
Candidate's answer: {answer}

Give concise, specific, actionable feedback on this single answer, calibrated to what is
reasonably expected from a {level} candidate at {difficulty} difficulty.

{format_instructions}
"""
    ),
    input_variables=["interview_type", "level", "difficulty", "question", "answer"],
    partial_variables={"format_instructions": feedback_parser.get_format_instructions()},
)

evaluation_parser = PydanticOutputParser(pydantic_object=InterviewEvaluation)

interview_evaluation_prompt = PromptTemplate(
    template=(
        """You are an expert hiring evaluator. Review the full transcript of a {interview_type} interview
in {language}, for a {level} candidate at {difficulty} difficulty, targeting: {target_role}.

Full transcript:
{full_transcript}

Score the candidate holistically across all dimensions in the schema below, based on the entire
conversation, calibrated to what is reasonably expected from a {level} candidate.

{format_instructions}
"""
    ),
    input_variables=["interview_type", "language", "level", "difficulty", "target_role", "full_transcript"],
    partial_variables={"format_instructions": evaluation_parser.get_format_instructions()},
)


def _format_transcript(turns: List[InterviewTurn]) -> str:
    """Renders the Q/A history as plain text for injection into interview prompts."""
    if not turns:
        return "(no questions asked yet)"
    lines = []
    for i, t in enumerate(turns, 1):
        tag = " (follow-up)" if t.is_followup else ""
        lines.append(f"Q{i}{tag}: {t.question}")
        if t.answer:
            lines.append(f"A{i}: {t.answer}")
    return "\n".join(lines)


def _format_context_block(context: Dict[str, Any]) -> str:
    """Renders the loaded candidate/job/resume/portfolio/skills context as plain text for prompts."""
    if not context:
        return "(no additional context available)"
    lines = []
    for key in ["candidate", "job", "resume", "portfolio", "skills"]:
        section = context.get(key)
        if not section:
            continue
        if section.get("status") == "not_available":
            lines.append(f"- {key.capitalize()}: not available yet")
        else:
            lines.append(f"- {key.capitalize()}: {section}")
    return "\n".join(lines) if lines else "(no additional context available)"


def CreateInterview(
    candidate_id: str,
    interview_type: InterviewType,
    level: CandidateLevel,
    difficulty: Difficulty,
    language: InterviewLanguage = InterviewLanguage.EN,
    job_id: Optional[str] = None,
    target_role: Optional[str] = None,
    mode: InterviewMode = InterviewMode.TEXT,
    providers: Optional[ContextProviders] = None,
) -> dict:
    """
    Creates a new interview session: loads candidate / job / resume / portfolio /
    skills context, then generates the opening question.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        candidate_id (str)
        interview_type (InterviewType): HR | BEHAVIORAL | TECHNICAL | SITUATIONAL   (MVP scope)
        level (CandidateLevel): JUNIOR | MID | SENIOR
        difficulty (Difficulty): EASY | MEDIUM | HARD
        language (InterviewLanguage, default=EN): EN | AR
        job_id (str | None)
        target_role (str | None)
        mode (InterviewMode, default=TEXT): TEXT | VOICE | VIDEO
        providers (ContextProviders | None): loaders for candidate/job/resume/portfolio/skills.
          Defaults to safe stubs if a module isn't ready yet.
      output_schema:
        { "session": InterviewSession dict, "question": str }
      status: active
      example_request:
        CreateInterview("cand_123", InterviewType.TECHNICAL, CandidateLevel.MID,
                         Difficulty.MEDIUM, target_role="Backend Engineer")
      example_response:
        { "session": {...}, "question": "Can you walk me through a recent backend project you built?" }
    """
    providers = providers or DEFAULT_PROVIDERS

    context = {
        "candidate": providers.load_candidate(candidate_id),
        "job": providers.load_job(job_id) if job_id else None,
        "resume": providers.load_resume(candidate_id),
        "portfolio": providers.load_portfolio(candidate_id),
        "skills": providers.load_skills(candidate_id),
    }

    session = InterviewSession(
        interview_id=str(uuid.uuid4()),
        candidate_id=candidate_id,
        job_id=job_id,
        target_role=target_role,
        interview_type=interview_type,
        level=level,
        difficulty=difficulty,
        language=language,
        mode=mode,
        context=context,
    )

    prompt_str = initial_question_prompt.format(
        interview_type=session.interview_type.value,
        language=session.language.full_name,
        level=session.level.value,
        difficulty=session.difficulty.value,
        target_role=session.target_role or "Not specified",
        question_count_guide=QUESTION_COUNT_GUIDE[session.interview_type],
        context_block=_format_context_block(session.context),
    )
    result: InitialQuestion = invoke_with_schema(prompt_str, initial_question_parser)

    session.turns.append(InterviewTurn(question=result.question, is_followup=False))
    session.current_question_index = 0
    session.status = InterviewStatus.IN_PROGRESS

    return json.dumps({"session": session.model_dump(), "question": result.question}, ensure_ascii=False, indent=2, default=str)


def _turn_state(sess: InterviewSession) -> Dict[str, bool]:
    """Deterministic view of what the interview is still allowed to do next."""
    limits = INTERVIEW_LIMITS[sess.interview_type]
    main_asked = sum(1 for t in sess.turns if not t.is_followup)
    followups_on_current = 0
    for t in reversed(sess.turns):
        if not t.is_followup:
            break
        followups_on_current += 1
    has_room = len(sess.turns) < limits["max_total_turns"]
    can_followup = has_room and followups_on_current < limits["max_followups_per_main"]
    can_new_main = has_room and main_asked < limits["max_main"]
    return {
        "can_followup": can_followup,
        "can_new_main": can_new_main,
        "can_end": main_asked >= limits["min_main"] or not (can_followup or can_new_main),
    }


def _allowed_actions_text(state: Dict[str, bool]) -> str:
    lines = []
    if state["can_followup"]:
        lines.append("- ask ONE follow-up on the last answer (is_followup=true)")
    if state["can_new_main"]:
        lines.append("- ask the next main question (is_followup=false)")
    if state["can_end"]:
        lines.append("- end the interview (should_end=true, question=null)")
    else:
        lines.append("(ending is NOT allowed yet — the minimum number of main questions has not been reached)")
    return "\n".join(lines)


def _decision_allowed(decision: NextTurnDecision, state: Dict[str, bool]) -> bool:
    if decision.should_end or not decision.question:
        return state["can_end"]
    return state["can_followup"] if decision.is_followup else state["can_new_main"]


def _decide_next_turn(sess: InterviewSession, max_attempts: int = 3) -> NextTurnDecision:
    """The LLM proposes the next turn; code decides whether it's allowed. No LLM call at all once
    nothing is allowed, a disallowed choice is retried, and if it still fails the interview ends."""
    state = _turn_state(sess)
    end = NextTurnDecision(should_end=True, is_followup=False, question=None)
    if not (state["can_followup"] or state["can_new_main"]):
        return end

    allowed = _allowed_actions_text(state)
    for attempt in range(max_attempts):
        note = "" if attempt == 0 else "\nYour previous choice was NOT allowed. Pick only from the list above."
        prompt_str = interview_next_prompt.format(
            interview_type=sess.interview_type.value,
            language=sess.language.full_name,
            level=sess.level.value,
            difficulty=sess.difficulty.value,
            target_role=sess.target_role or "Not specified",
            question_count_guide=QUESTION_COUNT_GUIDE[sess.interview_type],
            context_block=_format_context_block(sess.context),
            transcript=_format_transcript(sess.turns),
            allowed_actions=allowed + note,
        )
        decision: NextTurnDecision = invoke_with_schema(prompt_str, next_turn_parser)
        if _decision_allowed(decision, state):
            return decision
    return end


def SubmitAnswer(session: dict, answer_text: str) -> dict:
    """
    Records the candidate's answer, scores it, updates the interview context,
    and generates the next question (follow-up or new) — or ends the interview.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        session (dict): InterviewSession dict from CreateInterview / previous SubmitAnswer.
        answer_text (str): candidate's answer (already transcribed if mode is voice/video).
      output_schema:
        {
          "session": InterviewSession dict (updated),
          "question": str | None,
          "is_followup": bool,
          "should_end": bool,
          "last_feedback": QuestionFeedback dict
        }
      status: active
      error_schema:
        ValueError: if the session isn't IN_PROGRESS, or has no open question to answer.
      example_request:
        SubmitAnswer(session, "I led a team of 4 engineers to migrate our monolith to microservices...")
      example_response:
        { "session": {...}, "question": "What was the hardest part of that migration?",
          "is_followup": true, "should_end": false, "last_feedback": {...} }
    """
    sess = InterviewSession(**session)

    if sess.status != InterviewStatus.IN_PROGRESS:
        raise ValueError(f"Cannot submit an answer to an interview with status={sess.status.value}.")
    if not sess.turns or sess.turns[-1].answer is not None:
        raise ValueError("No open question to answer on this session.")

    current_turn = sess.turns[-1]
    current_turn.answer = answer_text

    fb_prompt = question_feedback_prompt.format(
        interview_type=sess.interview_type.value,
        level=sess.level.value,
        difficulty=sess.difficulty.value,
        question=current_turn.question,
        answer=answer_text,
    )
    current_turn.feedback = invoke_with_schema(fb_prompt, feedback_parser)


    decision: NextTurnDecision = _decide_next_turn(sess)

    if decision.should_end or not decision.question:
        sess.status = InterviewStatus.COMPLETED
    else:
        sess.turns.append(InterviewTurn(question=decision.question, is_followup=decision.is_followup))
        sess.current_question_index += 1

    return json.dumps({
        "session": sess.model_dump(),
        "question": decision.question,
        "is_followup": decision.is_followup,
        "should_end": sess.status == InterviewStatus.COMPLETED,
        "last_feedback": current_turn.feedback.model_dump(),
    }, ensure_ascii=False, indent=2, default=str)


def EndInterview(session: dict) -> dict:
    """
    Produces the final holistic evaluation and the Interview Report.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        session (dict): InterviewSession dict with one or more answered turns.
      output_schema:
        { "session": InterviewSession dict, "evaluation": InterviewEvaluation dict, "report": InterviewReport dict }
      status: active
      error_schema:
        ValueError: if the session has no answered turns to evaluate.
      example_request:
        EndInterview(session)
      example_response:
        { "session": {...}, "evaluation": { "overall_score": 82, ... }, "report": {...} }
    """
    sess = InterviewSession(**session)
    answered = [t for t in sess.turns if t.answer]
    if not answered:
        raise ValueError("Cannot evaluate an interview with no answered questions.")

    eval_prompt = interview_evaluation_prompt.format(
        interview_type=sess.interview_type.value,
        language=sess.language.full_name,
        level=sess.level.value,
        difficulty=sess.difficulty.value,
        target_role=sess.target_role or "Not specified",
        full_transcript=_format_transcript(sess.turns),
    )
    evaluation: InterviewEvaluation = invoke_with_schema(eval_prompt, evaluation_parser)

    sess.status = InterviewStatus.COMPLETED
    sess.evaluation = evaluation

    report = InterviewReport(
        interview_id=sess.interview_id,
        candidate_id=sess.candidate_id,
        target_role=sess.target_role,
        interview_type=sess.interview_type,
        level=sess.level,
        difficulty=sess.difficulty,
        language=sess.language,
        total_questions=len(sess.turns),
        total_answered=len(answered),
        evaluation=evaluation,
        per_question=sess.turns,
    )
    sess.report = report

    return json.dumps({
        "session": sess.model_dump(),
        "evaluation": evaluation.model_dump(),
        "report": report.model_dump(),
    }, ensure_ascii=False, indent=2, default=str)

In [ ]:
"""
Cell 11 — Skill Gap Engine

Independent from Resume Analysis: Resume Analysis extracts/scores the skills
already present in a CV; this engine answers a different question — given a
candidate's full profile (skills, experience, resume assessment, portfolio
review, interview results) and a target role or job, what's MISSING or WEAK
relative to that target, and how should closing the gap be prioritized?

Designed to be called from more than one flow (profile page, job-application
flow, post-interview report, career coaching) — the same way a
`POST /ai/skill-gap/analyze` endpoint would work: give it a candidate profile
plus a target_role and/or a job description, and optionally whatever
assessment signals already exist (ResumeEvaluation, PortfolioEvaluation,
EndInterview) — it composes all of them into one holistic gap analysis.
Every input besides `candidate` and (`target_role` / a job description) is
optional, so the engine degrades gracefully to whatever signals are actually
available.
"""


skill_gap_parser = PydanticOutputParser(pydantic_object=SkillGapAnalysisResult)

skill_gap_prompt = PromptTemplate(
    template=(
        """You are an expert technical career coach performing a Skill Gap Analysis.
Compare everything known about this candidate against the target role below, and
identify what is MISSING or WEAK — not what's already strong.

Target role: {target_role}

Candidate signals gathered so far:
{context_block}

Reference data (computed deterministically — a starting point, not ground truth;
correct or extend it using the narrative signals above):
- Current skills (from profile/projects): {current_skills_ref}
- Required/preferred skills (from job description, if any): {required_skills_ref}
- Keyword-matched skills: {matched_skills_ref}
- Keyword-missing skills: {missing_skills_ref}

Rules:
- currentSkills / requiredSkills / matchedSkills / missingSkills should reflect your own
  judgment, informed by (not copied verbatim from) the reference data above — merge synonyms,
  drop irrelevant matches, and add anything the keyword match missed.
- weakSkills are skills with SOME evidence (mentioned in resume/portfolio/interview) but that
  read as shallow, outdated, or underdeveloped for this role's seniority — never put a skill
  in both missingSkills and weakSkills.
- priority must cover every entry in missingSkills and weakSkills, each with a status,
  priority level, and a one-sentence reason grounded in the signals above.
- If a signal (resume, portfolio, interview) was not provided, do not penalize the candidate
  for its absence — reason only from what is actually given.

{format_instructions}
"""
    ),
    input_variables=[
        "target_role", "context_block",
        "current_skills_ref", "required_skills_ref",
        "matched_skills_ref", "missing_skills_ref",
    ],
    partial_variables={"format_instructions": skill_gap_parser.get_format_instructions()},
)


def _normalize_skills(skills: Optional[List[str]]) -> List[str]:
    """Lowercases, strips, and dedupes a skill list, preserving first-seen order."""
    return normalize_skills(skills)


def _extract_current_skills(candidate: Optional[dict]) -> List[str]:
    """Pulls current skills from a CandidateProfile dict: technical_skills + project tech stacks."""
    if not candidate:
        return []
    skills = list(candidate.get("technical_skills") or [])
    for project in candidate.get("projects") or []:
        skills.extend(project.get("tech_used") or [])
    return _normalize_skills(skills)


def _extract_required_skills(job_description: Optional[dict]) -> List[str]:
    """Pulls required + preferred skills from a JDAnalysisResult dict (IntakeJobDescription output)."""
    if not job_description:
        return []
    combined = list(job_description.get("required_skills") or []) + list(job_description.get("preferred_skills") or [])
    return _normalize_skills(combined)


def _format_skill_gap_context(
    candidate: Optional[dict],
    job_description: Optional[dict],
    resume_evaluation: Optional[dict],
    portfolio_evaluation: Optional[dict],
    interview_result: Optional[dict],
) -> str:
    """Renders every available signal into plain text for injection into the skill-gap prompt."""
    lines = []

    if candidate:
        lines.append(
            f"- Candidate: {candidate.get('years_of_experience', 'unknown')} yrs experience, "
            f"current role: {candidate.get('current_job_title') or 'not specified'}, "
            f"desired role: {candidate.get('desired_role') or 'not specified'}, "
            f"highest degree: {candidate.get('highest_degree') or 'not specified'}."
        )
    else:
        lines.append("- Candidate profile: not available.")

    if job_description:
        lines.append(
            f"- Job requirements ({job_description.get('job_title', 'target role')}): "
            f"experience level {job_description.get('experience_level') or 'unspecified'}, "
            f"responsibilities: {'; '.join(job_description.get('responsibilities') or []) or 'none listed'}."
        )
    else:
        lines.append("- Job description: not available — infer typical requirements for the target role.")

    if resume_evaluation:
        lines.append(
            f"- Resume assessment: score {resume_evaluation.get('score')}/100, "
            f"ATS score {resume_evaluation.get('atsScore')}/100, "
            f"missing skills flagged: {resume_evaluation.get('missingSkills') or []}, "
            f"weaknesses: {resume_evaluation.get('weaknesses') or []}."
        )
    else:
        lines.append("- Resume assessment: not available.")

    if portfolio_evaluation and portfolio_evaluation.get("status") != "UNSUPPORTED_SOURCE":
        lines.append(
            f"- Portfolio assessment: score {portfolio_evaluation.get('score')}/100, "
            f"missing elements: {portfolio_evaluation.get('missingElements') or []}, "
            f"weaknesses: {portfolio_evaluation.get('weaknesses') or []}."
        )
    else:
        lines.append("- Portfolio assessment: not available.")

    interview_evaluation = None
    if interview_result:
        interview_evaluation = interview_result.get("evaluation") or (interview_result.get("report") or {}).get("evaluation")
    if interview_evaluation:
        lines.append(
            f"- Interview results: overall {interview_evaluation.get('overall_score')}/100, "
            f"technical knowledge {interview_evaluation.get('technical_knowledge')}/100, "
            f"domain knowledge {interview_evaluation.get('domain_knowledge')}/100, "
            f"problem solving {interview_evaluation.get('problem_solving')}/100, "
            f"weaknesses: {interview_evaluation.get('weaknesses') or []}."
        )
    else:
        lines.append("- Interview results: not available.")

    return "\n".join(lines)


def AnalyzeSkillGap(
    candidate: dict,
    target_role: Optional[str] = None,
    job_description: Optional[dict] = None,
    job_description_text: Optional[str] = None,
    resume_evaluation: Optional[dict] = None,
    portfolio_evaluation: Optional[dict] = None,
    interview_result: Optional[dict] = None,
    max_retries: int = 2,
) -> dict:
    """
    Standalone Skill Gap Engine. Compares a candidate against a target role or job,
    across every assessment signal available, and returns missing/weak skills with
    priority and recommendations — reusable across profile, application, and
    post-interview flows (equivalent to `POST /ai/skill-gap/analyze`).

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        candidate (dict): CandidateProfile dict — output of ExtractCandidateProfile.
        target_role (str | None): Job title to evaluate against. Required if neither
          job_description nor job_description_text is given.
        job_description (dict | None): Pre-parsed JDAnalysisResult dict — output of
          IntakeJobDescription, if already parsed elsewhere. Takes priority over
          job_description_text if both are given.
        job_description_text (str | None): Raw job posting text — parsed internally
          via IntakeJobDescription if job_description is not already provided.
        resume_evaluation (dict | None): Output of ResumeEvaluation, if available.
        portfolio_evaluation (dict | None): Output of PortfolioEvaluation, if available.
        interview_result (dict | None): Output of EndInterview (the dict with
          "session"/"evaluation"/"report" keys), if the candidate has interviewed.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema: SkillGapAnalysisResult — target_role, currentSkills, requiredSkills,
        matchedSkills, missingSkills, weakSkills, priority (list of SkillPriority),
        readinessScore, summary, recommendations.
      status: active
      error_schema:
        ValueError: Raised if neither target_role, job_description, nor
          job_description_text is provided.
        RuntimeError: Raised after max_retries+1 failed LLM parse attempts.
      example_request:
        AnalyzeSkillGap(candidate_profile, target_role="Senior Frontend Developer",
                         resume_evaluation=cv_result, interview_result=end_interview_result)
      example_response:
        { "currentSkills": ["react", "javascript", "typescript"],
          "missingSkills": ["system design", "performance optimization"],
          "priority": [{"skill": "system design", "status": "missing",
                         "priority": "high", "reason": "..."}], "readinessScore": 62, ... }
    """
    if job_description is None and job_description_text:
        job_description = json.loads(IntakeJobDescription(job_description_text))

    if not target_role and job_description:
        target_role = job_description.get("job_title")

    if not target_role:
        raise ValueError(
            "AnalyzeSkillGap requires a target_role, or a job_description / "
            "job_description_text to derive one from."
        )

    current_skills_ref = _extract_current_skills(candidate)
    required_skills_ref = _extract_required_skills(job_description)
    matched_skills_ref = sorted(set(current_skills_ref) & set(required_skills_ref))
    missing_skills_ref = sorted(set(required_skills_ref) - set(current_skills_ref))

    context_block = _format_skill_gap_context(
        candidate, job_description, resume_evaluation, portfolio_evaluation, interview_result
    )

    prompt_str = skill_gap_prompt.format(
        target_role=target_role,
        context_block=context_block,
        current_skills_ref=current_skills_ref or "none extracted",
        required_skills_ref=required_skills_ref or "none extracted — infer typical requirements for this role",
        matched_skills_ref=matched_skills_ref or "none",
        missing_skills_ref=missing_skills_ref or "none",
    )

    result: SkillGapAnalysisResult = invoke_with_schema(prompt_str, skill_gap_parser, max_retries=max_retries)
    output = result.model_dump()
    for key in ("currentSkills", "requiredSkills", "matchedSkills", "missingSkills", "weakSkills"):
        output[key] = normalize_skills(output.get(key))
    output["weakSkills"] = [s for s in output["weakSkills"] if s not in output["missingSkills"]]
    for item in output.get("priority") or []:
        item["skill"] = normalize_skill(item.get("skill"))
    output["target_role"] = target_role
    return json.dumps(output, ensure_ascii=False, indent=2, default=str)

In [ ]:
"""
Cell 12 — Feature: Career Readiness Orchestrator

Consumes the outputs of every other evaluator — ResumeEvaluation, PortfolioEvaluation,
ProfileEvaluation, AnalyzeSkillGap, EndInterview — plus the CandidateProfile and an
optional job description, and produces ONE cross-source Career Readiness Report.
It never re-analyzes the Resume/Portfolio/Interview itself; it only aggregates,
normalizes, and reasons over the structured results those tools already produced.

NOTHING IS HARDCODED:
  - The set of readiness dimensions is whatever evidence actually exists for THIS
    candidate (no fixed dimension list, no fixed labels table).
  - The importance/weight of each dimension is decided by the LLM, per call, by
    reasoning over the actual target_role / job description and exactly the
    dimensions that have evidence — there is no static role->weights lookup table.
    (Same working principle as AnalyzeSkillGap: the LLM judges, the code only
    aggregates and does arithmetic.)
  - weight_overrides lets a caller pin explicit weights for a specific call, but
    that's a per-call input, not a built-in default.

Pipeline:
  1. Collect all candidate evidence          -> _collect_evidence
  2. Validate data completeness               -> _collect_evidence (dataCompleteness)
  3. Normalize the outputs                    -> _collect_dimension_scores
  4. Evaluate readiness dimensions (weights)   -> LLM (career_readiness_prompt)
  5. Calculate Overall Career Readiness        -> _apply_dimension_weights
  6. Identify critical gaps / weaknesses       -> LLM
  7. Generate prioritized recommendations      -> LLM
  8. Generate the final Readiness Report       -> CareerReadinessResult
  9. Generate next actions / improvement plan  -> LLM
"""




def _label(dimension_key: str) -> str:
    """Turns a dimension key into a human-readable label on the fly — no fixed lookup table."""
    return dimension_key.replace("_", " ").strip().title()


def _parse_min_years(text: Optional[str]) -> Optional[float]:
    """Extracts the first number out of a JD years_of_experience string like '2-4 years'."""
    if not text:
        return None
    match = re.search(r"(\d+(?:\.\d+)?)", text)
    return float(match.group(1)) if match else None


def _score_experience_fit(candidate: Optional[dict], job_description: Optional[dict]) -> Optional[int]:
    """Heuristic 0-100 score comparing candidate years_of_experience against the JD's requirement.
    Only produced when data actually supports it — absent otherwise, like every other dimension."""
    candidate_years = (candidate or {}).get("years_of_experience")
    if candidate_years is None:
        return None

    required_years = _parse_min_years((job_description or {}).get("years_of_experience"))
    if required_years is None:
        if candidate_years >= 6: return 90
        if candidate_years >= 3: return 75
        if candidate_years >= 1: return 55
        return 35

    diff = candidate_years - required_years
    if diff >= 2: return 95
    if diff >= 0: return 85
    if diff >= -1: return 65
    if diff >= -2: return 45
    return 25


def _collect_evidence(
    candidate, job_description, resume_evaluation, portfolio_evaluation,
    profile_evaluation, skill_gap_result, interview_result,
) -> Dict[str, bool]:
    """Steps 1-2: which evidence sources are present, for dataCompleteness."""
    return {
        "candidate_profile": bool(candidate),
        "job_description": bool(job_description),
        "resume_evaluation": bool(resume_evaluation),
        "portfolio_evaluation": bool(portfolio_evaluation and portfolio_evaluation.get("status") != "UNSUPPORTED_SOURCE"),
        "profile_evaluation": bool(profile_evaluation),
        "skill_gap_result": bool(skill_gap_result),
        "interview_result": bool(interview_result),
    }


def _collect_dimension_scores(
    candidate, job_description, resume_evaluation, portfolio_evaluation,
    profile_evaluation, skill_gap_result, interview_result,
) -> Dict[str, Optional[int]]:
    """Step 3: normalizes whatever evaluator outputs exist into 0-100 scores.
    A dimension simply doesn't appear (None) if its source evaluator wasn't run —
    nothing forces a fixed universe of dimensions to always be present."""
    interview_evaluation = None
    if interview_result:
        interview_evaluation = interview_result.get("evaluation") or (interview_result.get("report") or {}).get("evaluation")

    portfolio_score = None
    if portfolio_evaluation and portfolio_evaluation.get("status") != "UNSUPPORTED_SOURCE":
        portfolio_score = portfolio_evaluation.get("score")

    profile_score = None
    if profile_evaluation:
        vals = [v for v in (profile_evaluation.get("completenessScore"), profile_evaluation.get("qualityScore")) if v is not None]
        profile_score = round(sum(vals) / len(vals)) if vals else None

    technical_score = interview_evaluation.get("technical_knowledge") if interview_evaluation else None
    if technical_score is None and skill_gap_result:
        required = skill_gap_result.get("requiredSkills") or []
        matched = skill_gap_result.get("matchedSkills") or []
        if required:
            technical_score = min(100, round(len(matched) / len(required) * 100))

    skill_gap_readiness = skill_gap_result.get("readinessScore") if skill_gap_result else None

    return {
        "technical_skills": technical_score,
        "skill_gap_readiness": skill_gap_readiness,
        "resume_quality": resume_evaluation.get("score") if resume_evaluation else None,
        "portfolio": portfolio_score,
        "profile_quality": profile_score,
        "interview_performance": interview_evaluation.get("overall_score") if interview_evaluation else None,
        "communication": interview_evaluation.get("communication") if interview_evaluation else None,
        "experience_fit": _score_experience_fit(candidate, job_description),
    }


def _top(items, n: int = 3) -> List[str]:
    """First n non-empty items as stripped strings."""
    return [str(i).strip() for i in (items or []) if i and str(i).strip()][:n]


def _ev(label: str, items, n: int = 2) -> List[str]:
    return [f"{label}: {i}" for i in _top(items, n)]


def _collect_dimension_evidence(
    candidate, job_description, resume_evaluation, portfolio_evaluation,
    profile_evaluation, skill_gap_result, interview_result,
) -> Dict[str, List[str]]:
    """Short, source-tagged evidence per dimension, pulled straight from the evaluator
    outputs (no LLM call), so every score is traceable and nothing is invented."""
    interview_eval = None
    if interview_result:
        interview_eval = interview_result.get("evaluation") or (interview_result.get("report") or {}).get("evaluation")
    required = (skill_gap_result or {}).get("requiredSkills") or []
    matched = (skill_gap_result or {}).get("matchedSkills") or []
    ev: Dict[str, List[str]] = {}

    if resume_evaluation:
        ev["resume_quality"] = (
            [f"ATS match {resume_evaluation.get('atsScore')}/100"]
            + _ev("Strength", resume_evaluation.get("strengths"))
            + _ev("Weakness", resume_evaluation.get("weaknesses"))
            + _ev("Missing skill", resume_evaluation.get("missingSkills"), 3)
        )
    if portfolio_evaluation and portfolio_evaluation.get("status") != "UNSUPPORTED_SOURCE":
        ev["portfolio"] = (
            _ev("Strength", portfolio_evaluation.get("strengths"))
            + _ev("Weakness", portfolio_evaluation.get("weaknesses"))
            + _ev("Missing element", portfolio_evaluation.get("missingElements"), 3)
        )
    if profile_evaluation:
        ev["profile_quality"] = (
            [f"Completeness {profile_evaluation.get('completenessScore')}/100, quality {profile_evaluation.get('qualityScore')}/100"]
            + _ev("Missing field", profile_evaluation.get("missingFields"), 3)
            + _ev("Weak field", profile_evaluation.get("weakFields"), 3)
        )
    if interview_eval:
        ev["interview_performance"] = (
            _ev("Strength", interview_eval.get("strengths"))
            + _ev("Weakness", interview_eval.get("weaknesses"))
        )
        ev["communication"] = [
            f"Communication {interview_eval.get('communication')}, confidence {interview_eval.get('confidence')}, "
            f"language proficiency {interview_eval.get('language_proficiency')}"
        ]
    if interview_eval and interview_eval.get("technical_knowledge") is not None:
        ev["technical_skills"] = [
            f"Interview: technical {interview_eval.get('technical_knowledge')}, "
            f"domain {interview_eval.get('domain_knowledge')}, problem solving {interview_eval.get('problem_solving')}"
        ]
    elif required:
        ev["technical_skills"] = [f"Matched {len(matched)}/{len(required)} required skills"]
    if skill_gap_result:
        ev["skill_gap_readiness"] = (
            ([f"Matched {len(matched)}/{len(required)} required skills"] if required else [])
            + _ev("Missing skill", skill_gap_result.get("missingSkills"), 3)
            + _ev("Weak skill", skill_gap_result.get("weakSkills"), 3)
        )
    years = (candidate or {}).get("years_of_experience")
    if years is not None:
        req = _parse_min_years((job_description or {}).get("years_of_experience"))
        ev["experience_fit"] = [f"{years} yrs experience" + (f" vs {req:g}+ required" if req is not None else "")]
    return {k: v for k, v in ev.items() if v}


def _format_dimensions_for_prompt(
    available_dims: Dict[str, int], evidence: Optional[Dict[str, List[str]]] = None,
) -> str:
    """Lists exactly the dimensions that have evidence: the score plus the facts behind it."""
    evidence = evidence or {}
    out = []
    for key, score in available_dims.items():
        out.append(f"- {key} ({_label(key)}): {score}/100")
        out.extend(f"    · {e}" for e in evidence.get(key, []))
    return "\n".join(out)


def _format_readiness_context(candidate, job_description) -> str:
    """Only what the dimensions block can't carry: who the candidate is and what the role asks.
    Scores and evaluator findings live in dimensions_block, so nothing is repeated here."""
    lines = []
    if candidate:
        lines.append(
            f"- Candidate: {candidate.get('years_of_experience', 'unknown')} yrs experience, "
            f"current role: {candidate.get('current_job_title') or 'not specified'}, "
            f"desired role: {candidate.get('desired_role') or 'not specified'}, "
            f"highest degree: {candidate.get('highest_degree') or 'not specified'}."
        )
    if job_description:
        lines.append(
            f"- Role ({job_description.get('job_title', 'target role')}): "
            f"level {job_description.get('experience_level') or 'unspecified'}, "
            f"key responsibilities: {'; '.join((job_description.get('responsibilities') or [])[:5]) or 'none listed'}."
        )
    return "\n".join(lines) or "- No candidate or job context provided."



career_readiness_parser = PydanticOutputParser(pydantic_object=CareerReadinessNarrative)

career_readiness_prompt = PromptTemplate(
    template=(
        """You are an expert career coach synthesizing MULTIPLE independent assessments into one
holistic Career Readiness Report for a candidate targeting: {target_role_display}.
Mode: {readiness_mode} readiness evaluation.

Evidence gathered so far:
{context_block}

Readiness dimensions with evidence for THIS candidate (weight ONLY these — do not invent
or omit any, and do not weight anything not listed here):
{dimensions_block}

Your job:
1. Assign each listed dimension a weight (0-1) reflecting how much it genuinely matters
   for THIS specific target role — e.g. a hands-on engineering role should weight
   technical/portfolio/interview evidence heavily, while a client-facing or sales-type
   role should weight interview/communication/experience heavily. Base this on the
   target role and evidence above, not on any fixed formula. Weights should sum to ~1.0.
2. Identify the 2-5 most critical gaps blocking readiness (most severe first).
3. Surface genuine strengths grounded in the evidence above.
4. Give prioritized, actionable recommendations.
5. Build a concrete next-actions improvement plan (immediate / short_term / long_term).
6. Write a short overall narrative summary.

{format_instructions}"""
    ),
    input_variables=["target_role_display", "readiness_mode", "context_block", "dimensions_block"],
    partial_variables={"format_instructions": career_readiness_parser.get_format_instructions()},
)


def _validate_weight_overrides(
    weight_overrides: Optional[Dict[str, float]], available_dims: Dict[str, int],
) -> Dict[str, float]:
    """Validates per-call weight overrides before any LLM call is made.
    Keys with no evidence are ignored (with a warning); bad values raise ValueError."""
    if not weight_overrides:
        return {}
    cleaned = {}
    for key, value in weight_overrides.items():
        if key not in available_dims:
            print(f"[EvaluateCareerReadiness] weight_overrides key '{key}' ignored — "
                  f"no evidence for it. Available: {list(available_dims)}")
            continue
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not (0 <= value <= 1):
            raise ValueError(f"weight_overrides['{key}'] must be a number between 0 and 1, got {value!r}.")
        cleaned[key] = float(value)
    if sum(cleaned.values()) > 1 + 1e-9:
        raise ValueError(f"weight_overrides must sum to at most 1.0, got {sum(cleaned.values()):.3f}.")
    return cleaned


def _apply_dimension_weights(
    available_dims: Dict[str, int],
    llm_weights: List[DimensionWeight],
    weight_overrides: Optional[Dict[str, float]] = None,
):
    """Step 5: merges the LLM's per-role weights with validated caller overrides, then
    computes the weighted score.
      - Overridden dimensions keep exactly their pinned share.
      - The remaining share (1 - sum of pins) is split across the other dimensions in
        proportion to the LLM's weights.
      - Unknown, duplicate, or invalid LLM entries are ignored; a dimension the LLM skipped
        gets the average LLM weight. An explicit LLM weight of 0 is respected."""
    overrides = weight_overrides or {}

    llm_map, rationale_map, seen = {}, {}, set()
    for w in llm_weights or []:
        if w.dimension not in available_dims or w.dimension in seen:
            continue
        seen.add(w.dimension)
        rationale_map[w.dimension] = w.rationale
        if w.weight is not None and 0 <= w.weight < float("inf"):
            llm_map[w.dimension] = float(w.weight)

    free_keys = [k for k in available_dims if k not in overrides]
    fallback = (sum(llm_map.values()) / len(llm_map)) if llm_map else 1.0
    free_raw = {k: llm_map.get(k, fallback) for k in free_keys}
    free_total = sum(free_raw.values())
    if free_keys and free_total <= 0:
        free_raw, free_total = {k: 1.0 for k in free_keys}, float(len(free_keys))

    free_share = 1.0 - sum(overrides.values())
    final = dict(overrides)
    for k in free_keys:
        final[k] = free_share * free_raw[k] / free_total

    total_weight = sum(final.values())
    if total_weight <= 0:
        final, total_weight = {k: 1.0 for k in available_dims}, float(len(available_dims))

    dimensions, weighted_sum = [], 0.0
    for key, score in available_dims.items():
        normalized_weight = final[key] / total_weight
        weighted_sum += score * normalized_weight
        dimensions.append(ReadinessDimension(
            name=_label(key),
            score=score,
            weight=round(normalized_weight, 3),
            rationale=(
                "Caller-provided override for this call." if key in overrides
                else rationale_map.get(key, "No rationale provided.")
            ),
        ))
    return round(weighted_sum), dimensions


def _readiness_level(overall_score: int) -> str:
    """Generic 0-100 banding — a fixed scale label, not a role-based assumption."""
    if overall_score >= 85: return "highly_ready"
    if overall_score >= 70: return "ready"
    if overall_score >= 50: return "developing"
    if overall_score >= 30: return "emerging"
    return "not_ready"


def EvaluateCareerReadiness(
    candidate: dict,
    target_role: Optional[str] = None,
    job_description: Optional[dict] = None,
    job_description_text: Optional[str] = None,
    resume_evaluation: Optional[dict] = None,
    portfolio_evaluation: Optional[dict] = None,
    profile_evaluation: Optional[dict] = None,
    skill_gap_result: Optional[dict] = None,
    interview_result: Optional[dict] = None,
    weight_overrides: Optional[Dict[str, float]] = None,
    max_retries: int = 2,
) -> dict:
    """
    Career Readiness Orchestrator (fully dynamic). Aggregates the outputs of
    ResumeEvaluation, PortfolioEvaluation, ProfileEvaluation, AnalyzeSkillGap and
    EndInterview — plus the CandidateProfile and an optional job description — into one
    cross-source Career Readiness Report. Never re-runs Resume/Portfolio/Interview
    analysis itself, and never assumes a fixed dimension set or a fixed role->weights
    table: both the dimensions considered and their importance are derived from
    whatever evidence is actually passed in and the LLM's per-call reasoning about the
    target role.

    DATA CONTRACT:
      version: "1.0"
      input_schema:
        candidate (dict): CandidateProfile dict — output of ExtractCandidateProfile.
        target_role (str | None): Used for General readiness + role-aware weighting.
          Derived from job_description if omitted.
        job_description (dict | None): Pre-parsed JDAnalysisResult dict. If given (or
          derivable from job_description_text), readiness_mode becomes "job_specific".
        job_description_text (str | None): Raw JD text, parsed internally if
          job_description is not already provided.
        resume_evaluation, portfolio_evaluation, profile_evaluation, skill_gap_result,
          interview_result (dict | None): Outputs of the respective standalone
          evaluators. Every one is optional — dimensions with no backing evidence simply
          don't appear in the output at all.
        weight_overrides (dict | None): Optional {dimension_key: weight} map for this
          call only. Each weight must be 0-1 and the pins must sum to <= 1.0 (ValueError
          otherwise). Pinned dimensions keep exactly that share; the LLM's weights split
          the remainder across the other dimensions. Keys with no evidence are ignored
          with a warning.
        max_retries (int, default=2): Extra LLM retry attempts on parse failure.
      output_schema: CareerReadinessResult — target_role, readiness_mode,
        overallReadinessScore, readinessLevel, dimensions (only ones with evidence),
        dataCompleteness, criticalGaps, strengths, recommendations, nextActions, summary.
      status: active
      example_request:
        EvaluateCareerReadiness(candidate_profile, target_role="Backend Engineer",
                                 resume_evaluation=cv_result, portfolio_evaluation=portfolio_result,
                                 profile_evaluation=profile_result, skill_gap_result=gap_result,
                                 interview_result=end_interview_result)
      example_response:
        { "overallReadinessScore": 74, "readinessLevel": "ready",
          "dimensions": [{"name": "Interview Performance", "score": 80, "weight": 0.32, ...}],
          "criticalGaps": [...], "nextActions": [...], ... }
    """
    if job_description is None and job_description_text:
        job_description = json.loads(IntakeJobDescription(job_description_text))

    if not target_role and job_description:
        target_role = job_description.get("job_title")

    readiness_mode = "job_specific" if job_description else "general"
    data_completeness = _collect_evidence(
        candidate, job_description, resume_evaluation, portfolio_evaluation,
        profile_evaluation, skill_gap_result, interview_result,
    )

    dimension_scores = _collect_dimension_scores(
        candidate, job_description, resume_evaluation, portfolio_evaluation,
        profile_evaluation, skill_gap_result, interview_result,
    )
    available_dims = {k: v for k, v in dimension_scores.items() if v is not None}
    if not available_dims:
        raise ValueError("EvaluateCareerReadiness requires at least one evaluator result with data.")

    weight_overrides = _validate_weight_overrides(weight_overrides, available_dims)

    context_block = _format_readiness_context(candidate, job_description)

    prompt_str = career_readiness_prompt.format(
        target_role_display=target_role or "a general professional role",
        readiness_mode=readiness_mode,
        context_block=context_block,
        dimensions_block=_format_dimensions_for_prompt(
            available_dims,
            _collect_dimension_evidence(
                candidate, job_description, resume_evaluation, portfolio_evaluation,
                profile_evaluation, skill_gap_result, interview_result,
            ),
        ),
    )
    narrative: CareerReadinessNarrative = invoke_with_schema(prompt_str, career_readiness_parser, max_retries=max_retries)

    overall_score, dimensions = _apply_dimension_weights(
        available_dims, narrative.dimensionWeights, weight_overrides
    )
    readiness_level = _readiness_level(overall_score)

    result = CareerReadinessResult(
        target_role=target_role,
        readiness_mode=readiness_mode,
        overallReadinessScore=overall_score,
        readinessLevel=readiness_level,
        dimensions=dimensions,
        dataCompleteness=data_completeness,
        criticalGaps=narrative.criticalGaps,
        strengths=narrative.strengths,
        recommendations=narrative.recommendations,
        nextActions=narrative.nextActions,
        summary=narrative.summary,
    )
    return result.model_dump_json(indent=2)